In [3]:
import arxiv
import pandas as pd

client = arxiv.Client()
search = arxiv.Search(
    query="retrieval augmented generation",
    max_results=10,
    sort_by=arxiv.SortCriterion.Relevance
)

papers = []
for r in client.results(search):
    papers.append({
        "title": r.title,
        "authors": ", ".join(a.name for a in r.authors[:3]),
        "published": r.published.date(),
        "pdf_url": r.pdf_url,
    })

df = pd.DataFrame(papers)
df

,title,authors,published,pdf_url
0,AR-RAG: Autoregressive Retrieval Augmentation ...,"Jingyuan Qi, Zhiyang Xu, Qifan Wang",2025-06-08,https://arxiv.org/pdf/2506.06962v3
1,Intelligent Interaction Strategies for Context...,"Xiangrong, Zhu, Yuan Xu",2025-04-18,https://arxiv.org/pdf/2504.13684v1
2,Factually: Exploring Wearable Fact-Checking fo...,"Chitralekha Gupta, Hanjun Wu, Praveen Sasikumar",2025-04-24,https://arxiv.org/pdf/2504.17204v1
3,Designing AI Systems that Augment Human Perfor...,"Katelyn Xiaoying Mei, Nic Weber",2025-04-20,https://arxiv.org/pdf/2504.14689v1
4,Automated Literature Review Using NLP Techniqu...,"Nurshat Fateh Ali, Md. Mahdi Mohtasim, Shakil ...",2024-11-27,https://arxiv.org/pdf/2411.18583v1
5,EVOR: Evolving Retrieval for Code Generation,"Hongjin Su, Shuyang Jiang, Yuhang Lai",2024-02-19,https://arxiv.org/pdf/2402.12317v2
6,Riddle Me This! Stealthy Membership Inference ...,"Ali Naseh, Yuefeng Peng, Anshuman Suri",2025-02-01,https://arxiv.org/pdf/2502.00306v2
7,Ragas: Automated Evaluation of Retrieval Augme...,"Shahul Es, Jithin James, Luis Espinosa-Anke",2023-09-26,https://arxiv.org/pdf/2309.15217v2
8,MUST-RAG: MUSical Text Question Answering with...,"Daeyong Kwon, SeungHeon Doh, Juhan Nam",2025-07-31,https://arxiv.org/pdf/2507.23334v2
9,Engineering the RAG Stack: A Comprehensive Rev...,"Dean Wampler, Dave Nielson, Alireza Seddighi",2025-11-07,https://arxiv.org/pdf/2601.05264v1


In [8]:
import os
import urllib.request
import pymupdf 

os.makedirs("../data/papers", exist_ok=True)

first = papers[0]
filename = first["pdf_url"].split("/")[-1] + ".pdf"
filepath = os.path.join("../data/papers", filename)

if not os.path.exists(filepath):
    urllib.request.urlretrieve(first["pdf_url"], filepath)
print("File:", filepath)
print("Size:", os.path.getsize(filepath), "bytes")

doc = pymupdf.open(filepath)
text = "".join(page.get_text() for page in doc)
print("Pages:", len(doc))
print("Characters:", len(text))

File: ../data/papers/2506.06962v3.pdf
Size: 8870231 bytes
Pages: 18
Characters: 68775


In [9]:
def chunk_text(text, size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

chunks = chunk_text(text)
print("Total chunks:", len(chunks))
print("First chunk:")
print(chunks[0])

Total chunks: 172
First chunk:
arXiv:2506.06962v3  [cs.CV]  14 Jun 2025
AR-RAG: Autoregressive Retrieval Augmentation for
Image Generation
Jingyuan Qi* 1
Zhiyang Xu* 1
Qifan Wang2
Lifu Huang3
1Virginia Tech
2Meta
3 UC Davis
jingyq1@vt.edu
(a) Vanilla Image Generation
Prompt
(c) Patch-based Autoregressive Retrieval Augmentation (Ours)
...
Augmentation
Generation
Prompt
Generated Image
Generated Image
(b) Image-Based Retrieval Augmentation
Prompt
Generated Image
Retrieved Images
...
Query
Key
Query
Key
Query
Value
Key
Retrieval


In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
#Load the SBERT model
model=SentenceTransformer("all-MiniLM-L6-v2")
#embed all chunks 
embeddings=model.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)
# Build a FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension) 
index.add(np.array(embeddings, dtype="float32"))
print("Index size:", index.ntotal)


Batches: 100%|███████████████████████████████████| 6/6 [00:23<00:00,  3.96s/it]

Embeddings shape: (172, 384)
Index size: 172


In [11]:
def search(query, k=3):
    q_emb = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_emb, dtype="float32"), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "chunk": chunks[idx]
        })
    return results

query = "autoregressive retrieval augmentation for image generation"
results = search(query, k=3)

for i, r in enumerate(results, 1):
    print(f"--- Result {i} (score: {r['score']:.3f}) ---")
    print(r["chunk"][:300])
    print()

--- Result 1 (score: 0.813) ---
can dynamically adjust retrievals based on evolving generation states.
7
Conclusion
In this work, we propose Autoregressive Retrieval Augmentation (AR-RAG), a novel retrieval
paradigm that enhances image synthesis by leveraging k-nearest neighbor retrievals at the patch level.
Unlike traditional ima

--- Result 2 (score: 0.806) ---
ourney-30K 2, Geneval [14], and DPG-Bench [18]. Experimental results
demonstrate that both DAiD and FAiD significantly improve the coherence and naturalness of
generated images while introducing only marginal computational overhead.
The contributions of our work can be summarized as follows:
• We pr

--- Result 3 (score: 0.775) ---
ation of unrelated visual elements. For example, as shown in
Figure 1(b), a basketball player present in the retrieved references, despite being irrelevant to the
input prompt, unintentionally appears in the generated image.
In this paper, we propose Autoregressive Retrieval Augmentation (AR-RAG), a

In [12]:
import pickle
import faiss

with open("../data/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

faiss.write_index(index, "../data/faiss.index")

with open("../data/meta.pkl", "wb") as f:
    pickle.dump([{"text": c, "source": filename} for c in chunks], f)

print("Saved chunks:", len(chunks))
print("Saved index:", index.ntotal)

Saved chunks: 172
Saved index: 172


In [13]:
import os
print(os.listdir("../data"))

['chunks.pkl', 'faiss.index', 'meta.pkl', 'papers']


In [16]:
query = "What is autoregressive retrieval augmentation?"
results = search(query, k=5)

for i, r in enumerate(results, 1):
    print(f"--- {i}. score={r['score']:.3f} ---")
    print(r["chunk"][:200])
    print()

--- 1. score=0.606 ---
3] and our AR-RAG. ImageRAG excessively copies
retrieved images and does not follow user prompts.
Figure 5 presents a comparative analysis of conventional image-level and our autoregressive patch-
lev

--- 2. score=0.559 ---
arXiv:2506.06962v3  [cs.CV]  14 Jun 2025
AR-RAG: Autoregressive Retrieval Augmentation for
Image Generation
Jingyuan Qi* 1
Zhiyang Xu* 1
Qifan Wang2
Lifu Huang3
1Virginia Tech
2Meta
3 UC Davis
jingyq1

--- 3. score=0.543 ---
se for Janus-pro which is detailed in Section 3.1.
3
AR-RAG: Patch-based Autoregressive Retrieval Augmentation
3.1
Patch-based Retrieval Database Construction
We build a patch-based retrieval database

--- 4. score=0.542 ---
lue line) compared
to randomly sampled tokens (red dashed line). The
curved arrow indicates a broken y-axis that accom-
modates the large gap between the retrieved token
and the random token baseline.

--- 5. score=0.536 ---
rieved values, and
gray boxes with the question mark are next image patches t

In [17]:
def ask(query, k=5):
    results = search(query, k=k)
    context = "\n\n".join(
        f"[Chunk {i+1}, score {r['score']:.3f}]\n{r['chunk']}"
        for i, r in enumerate(results)
    )
    prompt = f"""You are a research assistant. Answer the question using the context below.
Synthesize information across chunks if needed. If the context truly doesn't
contain relevant information, say so. Cite which chunk(s) you used.

Context:
{context}

Question: {query}

Answer:"""
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    return response.text

answer = ask("What is autoregressive retrieval augmentation?")
print(answer)

Based on the provided context, **Autoregressive Retrieval Augmentation (AR-RAG)** is a paradigm for image generation that dynamically enhances the generation process by autoregressively incorporating $k$-nearest neighbor retrievals at the patch (or image token) level [Chunk 4, Chunk 5]. 

Key aspects include:
* **Patch-Level vs. Image-Level Retrieval:** Unlike conventional retrieval-augmented approaches that perform a single, static retrieval before generation and condition the entire output on fixed reference images (which often leads to overcopying irrelevant visual content), AR-RAG retrieves relevant patches during the generation sequence [Chunk 1, Chunk 5].
* **Database & Mechanism:** It retrieves top-$K$ image tokens/patches from a patch-based retrieval database constructed from large-scale image datasets to assist the autoregressive model in predicting the next image patches [Chunk 3, Chunk 4, Chunk 5].

**Sources:** [Chunk 1], [Chunk 3], [Chunk 4], [Chunk 5]
